In [ ]:
from IPython.display import HTML
HTML(open('../style.css').read())

In [ ]:
from typing import TypeVar

# Converting an <span style="font-variant:small-caps;">Nfa</span> into a <span style="font-variant:small-caps;">Dfa</span>

## Type Checking

The functions in this notebook carry *type annotations*.  *Python* itself ignores these annotations, but the
type checker [*basedpyright*](https://docs.basedpyright.com) can use them to find errors before the program is
run.  In *JupyterLab*, the extension *jupyterlab-lsp* runs *basedpyright* in the background and underlines
type errors while you type.  On the command line, the command
```
basedpyright 01-NFA-2-DFA.ipynb
```
checks the whole notebook.  The settings of the type checker are stored in the file `pyrightconfig.json` in the
directory `Python`.

Both packages, `basedpyright` and `jupyterlab-lsp`, are installed by the script `fl.sh`.  Start `jupyter lab` in the directory `Python`, so that the settings in `pyrightconfig.json` are used.

In this notebook we show how an <span style="font-variant:small-caps;">Nfa</span>
$$ F = \langle Q, \Sigma, \delta, q_0, A \rangle $$
can be transformed into a <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$ such that both finite state machines accept the
same language, that is we have
$$ L(F) = L\bigl(\texttt{det}(F)\bigr). $$
The idea behind this transformation is that the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$ has to 
compute the set of all states that the <span style="font-variant:small-caps;">Nfa</span> $F$ could be in. 
Hence the states of the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$ are 
**sets** of states of the <span style="font-variant:small-caps;">Nfa</span> $F$.  A set of these states contains all those states that the <span style="font-variant:small-caps;">Nfa</span> 
$F$ could have reached.  Furthermore, a set $M$ of states of the <span style="font-variant:small-caps;">Nfa</span> $F$ is an accepting state of the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$ if the set $M$ contains an accepting state of the <span style="font-variant:small-caps;">Nfa</span> $F$.

## Declaring the Necessary Types

`State` is the abstract type of the states of a finite state machine.

In [ ]:
State = TypeVar('State')

In [ ]:
Char = str

We represent the non-deterministic transition relation $\delta$ by the following type:

In [ ]:
TransRel = dict[tuple[State, Char], set[State]]

The type of the deterministic transition relation is more complicated, since the states of the <span style="font-variant:small-caps;">Dfa</span> are sets of states of the <span style="font-variant:small-caps;">Nfa</span>.

In [ ]:
TransRelDet = dict[tuple[frozenset[State], Char], frozenset[State]]

A non-deterministic finite state machine has the following type:

In [ ]:
NFA = tuple[frozenset[State], set[Char], TransRel, State, frozenset[State]]

The deterministic finite state machines produced in this notebook have the following type.

In [ ]:
DFA = tuple[set[frozenset[State]], set[Char], TransRelDet, frozenset[State], set[frozenset[State]]]

<hr style="height:5px;background-color:blue">

In order to present the construction of $\texttt{det}(F)$ we first have to define five auxiliary functions.
We start with the function `bigUnion`.  Given a set `M` that contains frozensets, the expression `bigUnion(M)`
returns the union of all sets in `M`, i.e. we have
$$ \texttt{bigUnion}(M) = \bigcup M = \bigl\{ x \bigm| \exists A \in M: x \in A \bigr\}. $$
The resulting set is returned as a `frozenset`.

**Function `bigUnion(M)`**
- *Input:* `M` is a set of frozensets.
- *Output:* The union of all sets in `M` as a `frozenset`.

In [ ]:
def bigUnion[T](M: set[frozenset[T]]) -> frozenset[T]:
    return frozenset({ x for A in M 
                         for x in A 
                     })

In [ ]:
bigUnion({ frozenset({1,2,3}), frozenset({2,3,4}), frozenset({3,4,5}) })

The function `epsClosure` takes two arguments:
- `s` is a state, 
- `δ` is the transition function of the <span style="font-variant:small-caps;">Nfa</span> $F$.

The function computes the set of all those states that can be reached from the state
`s` via $\varepsilon$-transitions.
Formally, the set $\texttt{epsClosure}(q)$ is defined inductively:
- $s \in \texttt{epsClosure}(s)$.
- $p \in \texttt{epsClosure}(s) \wedge r \in \delta(p, \varepsilon) \;\rightarrow\; r \in \texttt{epsClosure}(s)$.
 
  If the state $p$ is an element of the $\varepsilon$-closure of the state $s$ 
  and there is an $\varepsilon$-transition from $p$ to some state $r$, then $r$ 
  is also an element of the $\varepsilon$-closure of $s$.
  
The implementation of `epsClosure` uses a *fixed-point algorithm*.

**Function `epsClosure(s, δ)`**
- *Input:* `s` is a state of an <span style="font-variant:small-caps;">Nfa</span> and `δ` is its transition function.
- *Output:* The set of all states that can be reached from `s` via $\varepsilon$-transitions.

In [ ]:
def epsClosure(s: State, δ: TransRel) -> frozenset[State]:
    Result = frozenset({ s })
    while True:
        NewStates = bigUnion({ frozenset(δ.get((q, '𝜀'), set())) for q in Result })
        if NewStates <= Result:
            return Result
        Result |= NewStates

In order to transform an <span style="font-variant:small-caps;">Nfa</span> $F$ into a 
<span style="font-variant:small-caps;">Dfa</span>
$\texttt{det}(F)$ we have to extend the function $\delta:Q \times (\Sigma \cup \{\varepsilon\}) \rightarrow 2^Q$ into the function
$$\widehat{\delta}: Q \times \Sigma \rightarrow 2^Q. $$
The idea is that given a state $q$ and a character $c$,  the value of $\widehat{\delta}(q,c)$ is the set of all states that the
<span style="font-variant:small-caps;">Nfa</span> $F$ could reach when it reads the character $c$ in state $q$ and then performs an arbitrary number of $\varepsilon$-transitions.  Formally, the definition of $\widehat{\delta}$ is as follows:
$$ \widehat{\delta}(q_1, c) := \bigcup \bigl\{ \texttt{epsClosure}(q_2) \bigm| q_2 \in \delta(q_1, c) \bigr \}. $$
This formula is to be read as follows:
- For every state $q_2 \in Q$ that can be reached from the state $q_1$ by reading the character $c$ we
  compute $\texttt{epsClosure}(q_2)$.
- Then we take the union of all these sets $\texttt{epsClosure}(q_2)$.

The function $\widehat{\delta}$ is implemented as the function `deltaHat`, which takes three arguments:
- `s` is a state,
- `c` is a character,
- `𝛿` is the transition function of a 
  <span style="font-variant:small-caps;">Nfa</span>.

This function computes the set of all those states that can be reached 
from `s` when we first have a transition from state `s` to some state `p` 
on reading the character `c` followed by any number of $\varepsilon$-transitions
starting in `p`.

**Function `deltaHat(s, c, δ)`**
- *Input:* `s` is a state, `c` is a character, and `δ` is the transition function of an <span style="font-variant:small-caps;">Nfa</span>.
- *Output:* The set $\widehat{\delta}(s, c)$ of all states that can be reached from `s` by reading `c` followed by any number of $\varepsilon$-transitions.

In [ ]:
def deltaHat(s: State, c: Char, δ: TransRel) -> frozenset[State]:
    return bigUnion({ epsClosure(q, δ) for q in δ.get((s, c), set()) })

The function  $\widehat{\delta}$ maps a state into a set of states.  Since the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$ uses sets of states of the <span style="font-variant:small-caps;">Nfa</span> $F$ as its states we need a function that maps sets of states of the <span style="font-variant:small-caps;">Nfa</span> $F$ into sets of states.  Hence we generalize 
the function $\widehat{\delta}$ to the function
$$ \Delta: 2^Q \times \Sigma \rightarrow 2^Q $$
such that for a set $M$ of states and a character $c$ the expression $\Delta(M, c)$
computes the set of all those states that the <span style="font-variant:small-caps;">Nfa</span> $F$ could be in if it is in a state from the set $M$, then
reads the character $c$, and finally makes some $\varepsilon$-transitions.
The formal definition is as follows: 
$$ \Delta(M,c) := \bigcup \bigl\{ \widehat{\delta}(q,c) \bigm| q \in M \bigr\}. $$
This formula is easy to understand:  For every state  $q \in M$ we compute the set of states that the
<span style="font-variant:small-caps;">Nfa</span> $F$ could be in after reading the character $c$ and doing some 
$\varepsilon$-transitions.  Then we take the union of these sets.

**Function `capitalDelta(M, c, δ)`**
- *Input:* `M` is a set of states, `c` is a character, and `δ` is the transition function of an <span style="font-variant:small-caps;">Nfa</span>.
- *Output:* The set $\Delta(M, c)$, i.e. the union of the sets $\widehat{\delta}(q, c)$ for all $q \in M$.

In [ ]:
def capitalDelta(M: frozenset[State], c: Char, δ: TransRel) -> frozenset[State]:
    return bigUnion({ deltaHat(q, c, δ) for q in M })

The function `allStates` takes three arguments:
- $Q$ is $\texttt{epsClosure}(q_0)$, where $q_0$ is the start state of the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$,
- $\delta$ is the transition function of the <span style="font-variant:small-caps;">Nfa</span> $F$, and
- $\Sigma$ is the alphabet of the <span style="font-variant:small-caps;">Nfa</span> $F$.

The function `allStates` computes the set of all states of the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$
that can be reached from the start state.

**Function `allStates(Q, δ, Σ)`**
- *Input:* `Q` is the start state of the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$, `δ` is the transition function of the <span style="font-variant:small-caps;">Nfa</span> $F$, and `Σ` is its alphabet.
- *Output:* The set of all states of $\texttt{det}(F)$ that can be reached from `Q`.

In [ ]:
def allStates(Q: frozenset[State], δ: TransRel, Σ: set[Char]) -> set[frozenset[State]]:
    Result = { Q }
    while True:
        NewStates = { capitalDelta(M, c, δ) for M in Result 
                                            for c in Σ
                    }
        if NewStates <= Result:
            return Result
        Result |= NewStates

Now we are ready to formally define how the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$
is computed from the <span style="font-variant:small-caps;">Nfa</span>
$F = \bigl\langle Q, \Sigma, \delta, q_0, A \bigr\rangle$.
We define: 
$$ \texttt{det}(F) := \bigl\langle \texttt{allStates}(\texttt{epsClosure}(q_0)), \Sigma, \Delta, \texttt{epsClosure}(q_0), \widehat{A} \bigr\rangle $$
where the components of this tuple are given as follows:
- The set of states of $\texttt{det}(F)$ is the set of all states that can be reached from the set $\texttt{epsClosure}(q_0)$.
- The input alphabet $\Sigma$ does not change when going from $F$ to $\texttt{det}(F)$.
  After all, the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$ has to recognize the same language as the non-deterministic
  <span style="font-variant:small-caps;">Nfa</span> $F$.
- The function $\Delta$, that has been defined previously, specifies how the set of states changes when a
  character is read.
- The start state $\texttt{epsClosure}(q_0)$ of the <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$ is the set of all states
  that can be reached from the start state $q_0$ of the <span style="font-variant:small-caps;">Nfa</span> $F$
  via $\varepsilon$-transitions.
- The set of accepting states $\widehat{A}$ is the set of those subsets of $Q$ that contain an accepting
  state of the <span style="font-variant:small-caps;">Nfa</span> $F$:
  $$\widehat{A} := \bigl\{ M \in 2^Q \mid M \cap A \not= \{\} \bigr\}. $$

**Function `nfa2dfa(nfa)`**
- *Input:* `nfa` is an <span style="font-variant:small-caps;">Nfa</span> $F$.
- *Output:* The <span style="font-variant:small-caps;">Dfa</span> $\texttt{det}(F)$, which accepts the same language as $F$.

In [ ]:
def nfa2dfa(nfa: NFA) -> DFA:
    States, Σ, δ, q0, Final = nfa
    newStart  = epsClosure(q0, δ)
    NewStates = allStates(newStart, δ, Σ)
    newδ  = { (M, c): capitalDelta(M, c, δ) for M in NewStates
                                            for c in Σ
            }
    NewFinal = { M for M in NewStates if  M & Final != set() }
    return NewStates, Σ, newδ, newStart, NewFinal

To test this function, use the notebook `02-Test-NFA-2-DFA.ipynb`.